In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path("data/KAYA-Export-1781506842613.csv")

ROPE_LETTER_OFFSET = {"a": 0.0, "b": 0.25, "c": 0.5, "d": 0.75}


def parse_boulder_grade(grade: str) -> float:
    grade = grade.strip().lower()
    if grade == "vintro":
        return -1.0
    if grade == "vb":
        return 0.0
    match = re.fullmatch(r"v(\d+)", grade)
    if match:
        return float(match.group(1))
    raise ValueError(f"Unrecognized boulder grade: {grade}")


def parse_rope_grade(grade: str) -> float:
    grade = grade.strip()
    if grade.lower() == "5.intro":
        return 0.0
    match = re.fullmatch(r"5\.(\d+)([abcd])?", grade, flags=re.IGNORECASE)
    if not match:
        raise ValueError(f"Unrecognized rope grade: {grade}")
    base = float(match.group(1))
    suffix = match.group(2)
    return base + (ROPE_LETTER_OFFSET[suffix.lower()] if suffix else 0.0)


def grade_to_num(grade: str) -> float:
    grade = str(grade).strip()
    if grade.lower().startswith("v"):
        return parse_boulder_grade(grade)
    return parse_rope_grade(grade)


def infer_discipline(grade: str) -> str:
    return "boulder" if str(grade).strip().lower().startswith("v") else "rope"


df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(
    df["date"].str.replace(r" GMT\+0000 \(GMT\+00:00\)", "", regex=True),
    format="%a %b %d %Y %H:%M:%S",
)
df["grade_label"] = df["grade"].astype(str)
df["discipline"] = df["grade"].apply(infer_discipline)
df["grade_num"] = df["grade"].apply(grade_to_num)
df["is_send"] = df["ascent_type"].isin(["Onsight", "Flash", "Redpoint"])
df["is_onsight"] = df["ascent_type"].isin(["Onsight", "Flash"])
df["session_date"] = df["date"].dt.date
df["week"] = df["date"].dt.to_period("W").apply(lambda period: period.start_time)
df["month"] = df["date"].dt.to_period("M").dt.to_timestamp()

INTRO_GRADES = {"vIntro", "5.Intro"}
df_scored = df[~df["grade_label"].isin(INTRO_GRADES)].copy()

print(f"Loaded {len(df):,} ascents from {df['date'].min():%Y-%m-%d} to {df['date'].max():%Y-%m-%d}")
print(f"Discipline split: {df['discipline'].value_counts().to_dict()}")
print(f"Sessions: {df['session_date'].nunique():,}")

## Composite score

Each ascent gets a single **composite score** based on difficulty, style, and discipline. Higher is better.

**Formula:** `score = grade_num × ascent_multiplier × discipline_multiplier` (redpoints also ÷ √attempts)

| Factor | Weight |
|--------|--------|
| **Base** | Numeric grade (`v3` → 3, `5.11a` → 11.0, `5.11d` → 11.75) |
| **Onsight** | × 1.5 |
| **Flash** | × 1.4 |
| **Redpoint** | × 1.0, then ÷ √attempts if logged |
| **Repeat** | × 0.3 |
| **Rope** | × 1.2 (boulder × 1.0) |

**Examples**
- Onsight `5.11a` rope: 11.0 × 1.5 × 1.2 = **19.8**
- Onsight `v4` boulder: 4 × 1.5 × 1.0 = **6.0**
- Redpoint `v3` in 4 attempts: 3 × 1.0 × 1.0 ÷ 2 = **1.5**

**Chart lines**
- **Green (left axis):** 4-week rolling sum — recent climbing "output"
- **Purple dotted (right axis):** cumulative lifetime total
- **Orange dashed:** linear trend of the 4-week rolling score over time

Tweak weights in `SCORE_CONFIG` in the next cell.

In [ ]:
SCORE_CONFIG = {
    "ascent_multipliers": {
        "Onsight": 1.5,
        "Flash": 1.4,
        "Redpoint": 1.0,
        "Repeat": 0.3,
    },
    "discipline_multipliers": {
        "boulder": 1.0,
        "rope": 1.2,
    },
    "redpoint_attempt_penalty": True,
}


def compute_ascent_score(row: pd.Series) -> float:
    base = max(row["grade_num"], 0.0)
    ascent_multiplier = SCORE_CONFIG["ascent_multipliers"].get(row["ascent_type"], 0.0)
    discipline_multiplier = SCORE_CONFIG["discipline_multipliers"][row["discipline"]]
    score = base * ascent_multiplier * discipline_multiplier
    if (
        SCORE_CONFIG["redpoint_attempt_penalty"]
        and row["ascent_type"] == "Redpoint"
        and pd.notna(row["attempts"])
        and row["attempts"] > 0
    ):
        score /= np.sqrt(row["attempts"])
    return score


df["score"] = df.apply(compute_ascent_score, axis=1)

weekly_score = (
    df.groupby("week", as_index=False)
    .agg(weekly_score=("score", "sum"), weekly_routes=("grade", "count"))
    .sort_values("week")
)
weekly_score["rolling_4w_score"] = weekly_score["weekly_score"].rolling(4, min_periods=1).sum()
weekly_score["cumulative_score"] = weekly_score["weekly_score"].cumsum()

week_index = (weekly_score["week"] - weekly_score["week"].min()).dt.days.astype(float)
trend_coeffs = np.polyfit(week_index, weekly_score["rolling_4w_score"], 1)
weekly_score["rolling_4w_trend"] = np.polyval(trend_coeffs, week_index)
trend_slope_per_week = trend_coeffs[0] * 7

session_stats = (
    df.groupby("session_date", as_index=False)
    .agg(
        routes=("grade", "count"),
        sends=("is_send", "sum"),
        session_score=("score", "sum"),
    )
)
session_stats["send_rate"] = session_stats["sends"] / session_stats["routes"]

print(
    f"Avg routes/session: {session_stats['routes'].mean():.1f} | "
    f"Avg score/session: {session_stats['session_score'].mean():.1f} | "
    f"4-week score trend: {trend_slope_per_week:+.1f} pts/week"
)

In [ ]:
weekly_routes = (
    df.groupby(["week", "discipline"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
weekly_total = df.groupby("week", as_index=False).size().rename(columns={"size": "total_routes"})

weekly_sessions = (
    df.groupby("week", as_index=False)["session_date"]
    .nunique()
    .rename(columns={"session_date": "sessions"})
)

monthly_discipline = (
    df.groupby(["month", "discipline"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

monthly_ascent_type = (
    df.groupby(["month", "ascent_type"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

grade_order = (
    df.groupby(["grade_label", "discipline", "grade_num"], as_index=False)
    .size()
    .sort_values(["discipline", "grade_num"])
)
popular_grades = (
    df.groupby("grade_label")
    .size()
    .loc[lambda counts: counts >= 10]
    .index
)

onsight_source = df_scored[df_scored["grade_label"].isin(popular_grades)].copy()
onsight_by_grade_month = (
    onsight_source.groupby(["month", "grade_label", "discipline", "grade_num"], as_index=False)
    .agg(
        total=("grade_label", "count"),
        onsights=("is_onsight", "sum"),
    )
)
onsight_by_grade_month["onsight_rate"] = np.where(
    onsight_by_grade_month["total"] >= 3,
    onsight_by_grade_month["onsights"] / onsight_by_grade_month["total"],
    np.nan,
)
onsight_by_grade_month["hover"] = onsight_by_grade_month.apply(
    lambda row: f"{row['onsights']:.0f}/{row['total']:.0f}",
    axis=1,
)

sends_source = df_scored[df_scored["is_send"] & df_scored["grade_label"].isin(popular_grades)].copy()
sends_by_grade_month = (
    sends_source.groupby(["month", "grade_label", "discipline", "grade_num"], as_index=False)
    .size()
    .rename(columns={"size": "sends"})
)

all_sends_source = df_scored[df_scored["is_send"]].copy()
sends_by_grade_month_all = (
    all_sends_source.groupby(["month", "grade_label", "discipline", "grade_num"], as_index=False)
    .size()
    .rename(columns={"size": "sends"})
)

monthly_send_totals = (
    df_scored[df_scored["is_send"]]
    .groupby(["month", "discipline"], as_index=False)
    .size()
    .rename(columns={"size": "discipline_monthly_sends"})
)
send_share_by_grade_month = sends_by_grade_month_all.merge(
    monthly_send_totals,
    on=["month", "discipline"],
)
send_share_by_grade_month["send_share"] = (
    send_share_by_grade_month["sends"] / send_share_by_grade_month["discipline_monthly_sends"]
)
send_share_by_grade_month["hover"] = send_share_by_grade_month.apply(
    lambda row: f"{row['sends']:.0f}/{row['discipline_monthly_sends']:.0f} {row['discipline']}",
    axis=1,
)

recent_cutoff = df["date"].max() - pd.Timedelta(days=90)
grade_pyramid = (
    df.groupby(["grade_label", "discipline", "grade_num"], as_index=False)
    .agg(
        all_time=("grade_label", "count"),
        last_90_days=("date", lambda values: (values >= recent_cutoff).sum()),
    )
    .sort_values(["discipline", "grade_num"])
)

max_grade_timeline = (
    df[df["is_send"]]
    .groupby(["month", "discipline"], as_index=False)["grade_num"]
    .max()
)

gym_counts = (
    df.groupby(["gym", "discipline"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values("count", ascending=False)
)

session_calendar = (
    df.groupby("session_date", as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
session_calendar["session_date"] = pd.to_datetime(session_calendar["session_date"])
session_calendar["weekday"] = session_calendar["session_date"].dt.day_name()
session_calendar["week"] = session_calendar["session_date"].dt.isocalendar().week.astype(int)
session_calendar["year"] = session_calendar["session_date"].dt.year
session_calendar["year_week"] = (
    session_calendar["year"].astype(str)
    + "-W"
    + session_calendar["week"].astype(str).str.zfill(2)
)

redpoint_attempts = df[
    (df["ascent_type"] == "Redpoint") & df["attempts"].notna()
].copy()

daily_top3_score = (
    df.sort_values("score", ascending=False)
    .groupby(["session_date", "discipline"], as_index=False)
    .head(3)
    .groupby(["session_date", "discipline"], as_index=False)["score"]
    .mean()
    .rename(columns={"score": "top3_avg_score"})
)
daily_top3_score["session_date"] = pd.to_datetime(daily_top3_score["session_date"])

daily_top3_plot_parts = []
daily_top3_trend_slopes = {}
for discipline in ["boulder", "rope"]:
    sub = (
        daily_top3_score[daily_top3_score["discipline"] == discipline]
        .sort_values("session_date")
        .set_index("session_date")
        .copy()
    )
    sub["rolling_4w"] = sub["top3_avg_score"].rolling("28D", min_periods=1).mean()
    day_index = (sub.index - sub.index.min()).days.astype(float)
    trend_coeffs = np.polyfit(day_index, sub["rolling_4w"], 1)
    sub["trend"] = np.polyval(trend_coeffs, day_index)
    daily_top3_trend_slopes[discipline] = trend_coeffs[0] * 7
    sub["discipline"] = discipline
    daily_top3_plot_parts.append(sub.reset_index())
daily_top3_plot = pd.concat(daily_top3_plot_parts, ignore_index=True)

print(f"Prepared aggregations through {df['month'].max():%Y-%m}")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

COLORS = {
    "boulder": "#f97316",
    "rope": "#2563eb",
    "onsight": "#22c55e",
    "redpoint": "#ef4444",
    "flash": "#a855f7",
    "repeat": "#94a3b8",
}

fig = make_subplots(
    rows=3,
    cols=3,
    subplot_titles=(
        "Routes per week",
        "Discipline split (weekly)",
        "Sessions per week",
        "Onsight rate by grade (monthly)",
        "Ascent type mix (monthly)",
        "Composite score",
        "Grade pyramid",
        "Hardest sends per month",
        "Gym breakdown",
    ),
    specs=[
        [{"type": "bar"}, {"type": "scatter"}, {"type": "bar"}],
        [{"type": "heatmap"}, {"type": "bar"}, {"secondary_y": True}],
        [{"type": "bar"}, {"type": "scatter"}, {"type": "bar"}],
    ],
    vertical_spacing=0.08,
    horizontal_spacing=0.06,
)

fig.add_trace(
    go.Bar(
        x=weekly_total["week"],
        y=weekly_total["total_routes"],
        name="Routes/week",
        marker_color="#64748b",
        showlegend=False,
    ),
    row=1,
    col=1,
)

for discipline in ["boulder", "rope"]:
    subset = weekly_routes[weekly_routes["discipline"] == discipline]
    fig.add_trace(
        go.Scatter(
            x=subset["week"],
            y=subset["count"],
            mode="lines",
            stackgroup="discipline",
            name=discipline.title(),
            line=dict(width=0.6, color=COLORS[discipline]),
            fillcolor=COLORS[discipline],
        ),
        row=1,
        col=2,
    )

fig.add_trace(
    go.Bar(
        x=weekly_sessions["week"],
        y=weekly_sessions["sessions"],
        name="Sessions/week",
        marker_color="#0ea5e9",
        showlegend=False,
    ),
    row=1,
    col=3,
)

heatmap_grades = (
    onsight_by_grade_month[["grade_label", "discipline", "grade_num"]]
    .drop_duplicates()
    .sort_values(["discipline", "grade_num"])
)
grade_labels = heatmap_grades["grade_label"].tolist()
months = sorted(onsight_by_grade_month["month"].unique())
rate_matrix = []
hover_matrix = []
for grade_label in grade_labels:
    row_rates = []
    row_hover = []
    for month in months:
        cell = onsight_by_grade_month[
            (onsight_by_grade_month["grade_label"] == grade_label)
            & (onsight_by_grade_month["month"] == month)
        ]
        if cell.empty:
            row_rates.append(np.nan)
            row_hover.append("")
        else:
            row_rates.append(cell["onsight_rate"].iloc[0] * 100)
            row_hover.append(cell["hover"].iloc[0])
    rate_matrix.append(row_rates)
    hover_matrix.append(row_hover)

fig.add_trace(
    go.Heatmap(
        x=[pd.Timestamp(month).strftime("%Y-%m") for month in months],
        y=grade_labels,
        z=rate_matrix,
        text=hover_matrix,
        hovertemplate="Grade %{y}<br>Month %{x}<br>Onsight %{z:.0f}%<br>%{text}<extra></extra>",
        colorscale="Greens",
        zmin=0,
        zmax=100,
        colorbar=dict(title="Onsight %", len=0.25, y=0.52),
        showscale=True,
    ),
    row=2,
    col=1,
)

for ascent_type in ["Onsight", "Flash", "Redpoint", "Repeat"]:
    subset = monthly_ascent_type[monthly_ascent_type["ascent_type"] == ascent_type]
    fig.add_trace(
        go.Bar(
            x=subset["month"],
            y=subset["count"],
            name=ascent_type,
            marker_color=COLORS.get(ascent_type.lower(), "#64748b"),
        ),
        row=2,
        col=2,
    )

fig.add_trace(
    go.Scatter(
        x=weekly_score["week"],
        y=weekly_score["rolling_4w_score"],
        mode="lines",
        name="4-week rolling score",
        line=dict(color="#16a34a", width=2),
    ),
    row=2,
    col=3,
)
fig.add_trace(
    go.Scatter(
        x=weekly_score["week"],
        y=weekly_score["rolling_4w_trend"],
        mode="lines",
        name=f"Trend ({trend_slope_per_week:+.1f} pts/wk)",
        line=dict(color="#ea580c", width=2, dash="dash"),
    ),
    row=2,
    col=3,
)
fig.add_trace(
    go.Scatter(
        x=weekly_score["week"],
        y=weekly_score["cumulative_score"],
        mode="lines",
        name="Cumulative score",
        line=dict(color="#7c3aed", width=2, dash="dot"),
    ),
    row=2,
    col=3,
    secondary_y=True,
)

for discipline in ["boulder", "rope"]:
    subset = grade_pyramid[grade_pyramid["discipline"] == discipline]
    fig.add_trace(
        go.Bar(
            x=subset["grade_label"],
            y=subset["all_time"],
            name=f"{discipline.title()} all-time",
            marker_color=COLORS[discipline],
            opacity=0.45,
            legendgroup=f"pyramid-{discipline}",
            showlegend=False,
        ),
        row=3,
        col=1,
    )
    fig.add_trace(
        go.Bar(
            x=subset["grade_label"],
            y=subset["last_90_days"],
            name=f"{discipline.title()} last 90d",
            marker_color=COLORS[discipline],
            legendgroup=f"pyramid-{discipline}",
            showlegend=False,
        ),
        row=3,
        col=1,
    )

for discipline in ["boulder", "rope"]:
    subset = max_grade_timeline[max_grade_timeline["discipline"] == discipline]
    fig.add_trace(
        go.Scatter(
            x=subset["month"],
            y=subset["grade_num"],
            mode="lines+markers",
            name=f"Max {discipline}",
            line=dict(color=COLORS[discipline], width=2),
            marker=dict(size=6),
        ),
        row=3,
        col=2,
    )

top_gyms = gym_counts.groupby("gym")["count"].sum().sort_values(ascending=False).head(8).index.tolist()
gym_pivot = (
    gym_counts[gym_counts["gym"].isin(top_gyms)]
    .pivot(index="gym", columns="discipline", values="count")
    .fillna(0)
    .loc[top_gyms]
)
for discipline in ["boulder", "rope"]:
    if discipline in gym_pivot.columns:
        fig.add_trace(
            go.Bar(
                y=gym_pivot.index,
                x=gym_pivot[discipline],
                name=f"{discipline.title()} gym",
                orientation="h",
                marker_color=COLORS[discipline],
                showlegend=False,
            ),
            row=3,
            col=3,
        )

fig.update_layout(
    title="Climbing Log Dashboard — KAYA Export",
    template="plotly_white",
    hovermode="x unified",
    barmode="stack",
    height=2400,
    width=1600,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)

fig.update_xaxes(title_text="Week", row=1, col=1)
fig.update_xaxes(title_text="Week", row=1, col=2)
fig.update_xaxes(title_text="Week", row=1, col=3)
fig.update_xaxes(title_text="Month", row=2, col=2)
fig.update_xaxes(title_text="Week", row=2, col=3)
fig.update_xaxes(title_text="Grade", row=3, col=1)
fig.update_xaxes(title_text="Month", row=3, col=2)
fig.update_xaxes(title_text="Ascents", row=3, col=3)

fig.update_yaxes(title_text="Routes", row=1, col=1)
fig.update_yaxes(title_text="Routes", row=1, col=2)
fig.update_yaxes(title_text="Sessions", row=1, col=3)
fig.update_yaxes(title_text="Grade", row=2, col=1)
fig.update_yaxes(title_text="Ascents", row=2, col=2)
fig.update_yaxes(title_text="4-week score", row=2, col=3)
fig.update_yaxes(title_text="Cumulative score", row=2, col=3, secondary_y=True)
fig.update_yaxes(title_text="Count", row=3, col=1)
fig.update_yaxes(title_text="Grade index", row=3, col=2)

fig.show()

In [ ]:
import plotly.graph_objects as go

DISCIPLINE_COLORS = {"boulder": "#f97316", "rope": "#2563eb"}

top3_fig = go.Figure()
for discipline in ["boulder", "rope"]:
    subset = daily_top3_plot[daily_top3_plot["discipline"] == discipline]
    color = DISCIPLINE_COLORS[discipline]
    slope = daily_top3_trend_slopes[discipline]
    top3_fig.add_trace(
        go.Scatter(
            x=subset["session_date"],
            y=subset["top3_avg_score"],
            mode="markers",
            name=f"{discipline.title()} daily top-3",
            marker=dict(size=5, color=color, opacity=0.3),
        )
    )
    top3_fig.add_trace(
        go.Scatter(
            x=subset["session_date"],
            y=subset["rolling_4w"],
            mode="lines",
            name=f"{discipline.title()} 4-week avg",
            line=dict(color=color, width=2.5),
        )
    )
    top3_fig.add_trace(
        go.Scatter(
            x=subset["session_date"],
            y=subset["trend"],
            mode="lines",
            name=f"{discipline.title()} trend ({slope:+.2f}/wk)",
            line=dict(color=color, width=2, dash="dash"),
        )
    )

top3_fig.update_layout(
    title="Daily top-3 composite score (boulder vs rope)",
    template="plotly_white",
    height=600,
    width=1200,
    xaxis_title="Date",
    yaxis_title="Avg composite score (top 3 sends)",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)
top3_fig.show()

send_heatmap_grades = (
    sends_by_grade_month[["grade_label", "discipline", "grade_num"]]
    .drop_duplicates()
    .sort_values(["discipline", "grade_num"])
)
send_grade_labels = send_heatmap_grades["grade_label"].tolist()
send_months = sorted(sends_by_grade_month["month"].unique())
send_count_matrix = []
for grade_label in send_grade_labels:
    row_counts = []
    for month in send_months:
        cell = sends_by_grade_month[
            (sends_by_grade_month["grade_label"] == grade_label)
            & (sends_by_grade_month["month"] == month)
        ]
        row_counts.append(int(cell["sends"].iloc[0]) if not cell.empty else 0)
    send_count_matrix.append(row_counts)

sends_fig = go.Figure(
    data=go.Heatmap(
        x=[pd.Timestamp(month).strftime("%Y-%m") for month in send_months],
        y=send_grade_labels,
        z=send_count_matrix,
        colorscale="Blues",
        hovertemplate="Grade %{y}<br>Month %{x}<br>Sends %{z}<extra></extra>",
        colorbar=dict(title="Sends"),
    )
)
sends_fig.update_layout(
    title="Sends by grade (monthly)",
    template="plotly_white",
    height=800,
    width=1200,
    xaxis_title="Month",
    yaxis_title="Grade",
)
sends_fig.show()

send_share_heatmap_grades = (
    sends_by_grade_month_all[["grade_label", "discipline", "grade_num"]]
    .drop_duplicates()
    .sort_values(["discipline", "grade_num"])
)
send_share_grade_labels = send_share_heatmap_grades["grade_label"].tolist()
send_share_months = sorted(sends_by_grade_month_all["month"].unique())
send_share_matrix = []
send_share_hover = []
for grade_label in send_share_grade_labels:
    row_shares = []
    row_hover = []
    for month in send_share_months:
        cell = send_share_by_grade_month[
            (send_share_by_grade_month["grade_label"] == grade_label)
            & (send_share_by_grade_month["month"] == month)
        ]
        if cell.empty:
            row_shares.append(0.0)
            row_hover.append("")
        else:
            row_shares.append(cell["send_share"].iloc[0] * 100)
            row_hover.append(cell["hover"].iloc[0])
    send_share_matrix.append(row_shares)
    send_share_hover.append(row_hover)

send_share_fig = go.Figure(
    data=go.Heatmap(
        x=[pd.Timestamp(month).strftime("%Y-%m") for month in send_share_months],
        y=send_share_grade_labels,
        z=send_share_matrix,
        text=send_share_hover,
        colorscale="Purples",
        hovertemplate="Grade %{y}<br>Month %{x}<br>Share %{z:.1f}%<br>%{text}<extra></extra>",
        colorbar=dict(title="Share of sends %"),
    )
)
send_share_fig.update_layout(
    title="Share of monthly sends by grade (within boulder / rope)",
    template="plotly_white",
    height=800,
    width=1200,
    xaxis_title="Month",
    yaxis_title="Grade",
)
send_share_fig.show()

calendar_pivot = session_calendar.pivot_table(
    index="year_week",
    columns="weekday",
    values="count",
    aggfunc="sum",
    fill_value=0,
)
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
calendar_pivot = calendar_pivot.reindex(columns=weekday_order)

calendar_fig = go.Figure(
    data=go.Heatmap(
        x=calendar_pivot.columns,
        y=calendar_pivot.index,
        z=calendar_pivot.values,
        colorscale="Blues",
        hovertemplate="Week %{y}<br>%{x}<br>Ascents %{z}<extra></extra>",
        colorbar=dict(title="Ascents/day"),
    )
)
calendar_fig.update_layout(
    title="Session calendar heatmap",
    template="plotly_white",
    height=700,
    width=1200,
    xaxis_title="Weekday",
    yaxis_title="Year-week",
)
calendar_fig.show()

redpoint_grades = (
    redpoint_attempts.groupby("grade_label")["attempts"]
    .count()
    .loc[lambda counts: counts >= 5]
    .index
)
redpoint_box = redpoint_attempts[redpoint_attempts["grade_label"].isin(redpoint_grades)].copy()
grade_sort = (
    redpoint_box.groupby("grade_label")["grade_num"]
    .first()
    .sort_values()
    .index.tolist()
)

attempts_fig = go.Figure()
for grade_label in grade_sort:
    subset = redpoint_box[redpoint_box["grade_label"] == grade_label]
    attempts_fig.add_trace(
        go.Box(
            y=subset["attempts"],
            name=grade_label,
            marker_color="#ef4444",
            boxmean=True,
        )
    )

attempts_fig.update_layout(
    title="Redpoint attempts by grade",
    template="plotly_white",
    height=600,
    width=1200,
    xaxis_title="Grade",
    yaxis_title="Attempts to send",
    showlegend=False,
)
attempts_fig.show()

In [ ]:
def top_grade_label(discipline: str) -> str:
    subset = df[(df["discipline"] == discipline) & df["is_send"]]
    if subset.empty:
        return "n/a"
    return subset.sort_values("grade_num", ascending=False).iloc[0]["grade_label"]


busiest_month = (
    df.groupby("month").size().sort_values(ascending=False).head(1)
)
busiest_month_label = pd.Timestamp(busiest_month.index[0]).strftime("%Y-%m")
busiest_month_count = int(busiest_month.iloc[0])

most_used_gym = df["gym"].value_counts().idxmax()

latest_weeks = weekly_score.tail(8)
current_4w = latest_weeks["weekly_score"].tail(4).sum()
prior_4w = latest_weeks["weekly_score"].iloc[-8:-4].sum()
score_delta = current_4w - prior_4w

print("=== Climbing log summary ===")
print(f"Date range:        {df['date'].min():%Y-%m-%d} to {df['date'].max():%Y-%m-%d}")
print(f"Total ascents:     {len(df):,}")
print(f"Sessions:          {df['session_date'].nunique():,}")
print(f"Avg routes/session:{session_stats['routes'].mean():.1f}")
print()
print(f"Onsight rate (all):{df['is_onsight'].mean():.1%}")
print(f"  Boulder:         {df.loc[df['discipline'] == 'boulder', 'is_onsight'].mean():.1%}")
print(f"  Rope:            {df.loc[df['discipline'] == 'rope', 'is_onsight'].mean():.1%}")
print()
print(f"Top boulder send:  {top_grade_label('boulder')}")
print(f"Top rope send:     {top_grade_label('rope')}")
print()
print(f"Busiest month:     {busiest_month_label} ({busiest_month_count} ascents)")
print(f"Most-used gym:     {most_used_gym}")
print()
print(f"Last 4-week score: {current_4w:.1f}")
print(f"Prior 4-week score:{prior_4w:.1f}")
print(f"Score delta:       {score_delta:+.1f}")